In [ ]:
# 0,1: 00 for psi_1, 01 for psi_2 ...
# 2: For states
# 3,4: Ancillary qubits

# when theta = 32.56... 
# psi_1 = |0> 
# psi_2 = sqrt(2/3)|0> + sqrt(1/3)|1>
# psi_3 = sqrt(2/3)|0> - sqrt(1/3)|1>

In [ ]:
from qiskit import QuantumCircuit, ClassicalRegister
from qiskit.circuit import ParameterVector
from numpy import sqrt, array
from numpy.linalg import norm


def circuit_init():
    qc = QuantumCircuit(5)
    return qc

def PREP(qc):
    desired_vector = [sqrt(1/3), 0, sqrt(2)/3, 1/3, sqrt(2)/3, -1/3, 0, 0]
    qc.initialize(desired_vector, [2,1,0])
    return qc

def Unitary(qc, wires, num_layers):

    x = ParameterVector('x', 3 * num_layers)
    z = ParameterVector('z', 3 * num_layers)

    for i in range(num_layers):
        qc.barrier()
        qc.rx(x[3*i], wires[0])
        qc.rx(x[3*i+1], wires[1])
        qc.rx(x[3*i+2], wires[2])

        qc.rz(z[3*i], wires[0])
        qc.rz(z[3*i+1], wires[1])
        qc.rz(z[3*i+2], wires[2])
        
        # Apply CNOT gates between the qubits
        qc.cx(wires[0], wires[1])
        qc.cx(wires[1], wires[2]) 

    qc.measure_all()
    return qc

In [ ]:
from numpy.random import rand
num_layers=3
num_params= 6*num_layers
parmas_instant = rand(num_params)

qc = circuit_init()
qc = PREP(qc)
Unitary(qc, [2,3,4] , num_layers)
qc_reversed=qc.reverse_bits()
qc_reversed.draw(output='mpl', style = 'clifford') 


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel='ibm_quantum')
backend = service.least_busy(min_num_qubits=127)
print(backend)

pm = generate_preset_pass_manager(optimization_level=3,backend=backend)

candidate_circuit = pm.run(qc_reversed)
candidate_circuit.draw('mpl', fold=False, idle_wires=False)

In [ ]:
from qiskit.quantum_info import SparsePauliOp

# Define the factor (I + Z)/2 = |0><0|
o_term = SparsePauliOp.from_list([("I", 0.5), ("Z", 0.5)])

# Define the factor (I - Z)/2 = |1><1|
l_term = SparsePauliOp.from_list([("I", 0.5), ("Z", -0.5)])

I = SparsePauliOp.from_list([("I", 1)])

# Use tensor products to create (I + Z)/2 ⊗ (I + Z)/2 ⊗ (I + Z)/2 ⊗ (I + Z)/2
Ob1= o_term.tensor(o_term).tensor(I).tensor(o_term).tensor(o_term)
Ob2= o_term.tensor(l_term).tensor(I).tensor(o_term).tensor(l_term)
Ob3= l_term.tensor(o_term).tensor(I).tensor(l_term).tensor(o_term)

cost_hamiltonian = Ob1 + Ob2 + Ob3
cost_hamiltonian=cost_hamiltonian.apply_layout(candidate_circuit.layout)

In [ ]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    
    # Prepare the job input with the ansatz, Hamiltonian, and parameters
    pub = (ansatz, hamiltonian, params)
    job = estimator.run([pub])

    # Retrieve the result
    results = job.result()[0]

    # Extract the cost (expectation value)
    cost = results.data.evs
    print('cost is', cost)

    # Append cost to the global success probability list and return it
    success_probability.append(cost)
    return 1/cost

In [ ]:
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from scipy.optimize import minimize

init_params = rand(num_params)
success_probability = [] # Global variable

with Session(backend=backend) as session:

    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000
    estimator.options.max_execution_time = 4500

    # Set simple error suppression/mitigation options
    #estimator.options.dynamical_decoupling.enable = True
    #estimator.options.dynamical_decoupling.sequence_type = "XY4"
    #estimator.options.twirling.enable_gates = True
    #estimator.options.twirling.num_randomizations = "auto"

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-1,
    )
    print(result)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(success_probability, label="success probability")
plt.xlabel('Iteration')
plt.ylabel('Probability')
plt.legend()
plt.show()

In [ ]:
import openpyxl

# create a new workbook
workbook = openpyxl.Workbook()

# select the active worksheet
worksheet = workbook.active

# loop through the confidences array and write values to the worksheet
for i in range(len(success_probability)):
    worksheet.cell(row=i+1, column=1, value=float(success_probability[i]))

# save the workbook to a file
workbook.save('ME HEA L3 Sym states (strasbourg, tol01).xlsx')